# Preprocessing Pipeline
## ArtEmis Dataset - Image & Label Preparation

In [1]:
# Import
import os
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [2]:
# Load Data
df = pd.read_csv("../data/dataset_final.csv")
IMAGE_DIR = Path("../data/images")

print(f"Total samples: {len(df)}")

Total samples: 49056


In [5]:
# Label Encoding 
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])

print("Emotion: Label mapping:")
for emotion, label in zip(le.classes_, range(len(le.classes_))):
    print(f"  {emotion}: {label}")

Emotion: Label mapping:
  amusement: 0
  anger: 1
  awe: 2
  contentment: 3
  disgust: 4
  excitement: 5
  fear: 6
  sadness: 7


In [6]:
# Train/Val/Test Split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

Train: 39244
Val:   4906
Test:  4906


In [7]:
# Class Weights 
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights:")
for emotion, weight in zip(le.classes_, class_weights):
    print(f"  {emotion}: {weight:.4f}")

Class weights:
  amusement: 7.1094
  anger: 1.8567
  awe: 3.5807
  contentment: 1.4056
  disgust: 0.8456
  excitement: 4.4799
  fear: 0.5199
  sadness: 0.3332


In [8]:
# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Train transforms:", train_transforms)
print("\nVal/Test transforms:", val_test_transforms)

Train transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.7, 1.3), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Val/Test transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [9]:
# ArtEmisDataset
class ArtEmisDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.image_dir / row['filename']
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label

In [12]:
train_dataset = ArtEmisDataset(train_df, IMAGE_DIR, transform=train_transforms)
val_dataset   = ArtEmisDataset(val_df,   IMAGE_DIR, transform=val_test_transforms)
test_dataset  = ArtEmisDataset(test_df,  IMAGE_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify one batch loads correctly
images, labels = next(iter(train_loader))
print(f"\nBatch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")

Train batches: 1227
Val batches:   154
Test batches:  154

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])


## Preprocessing Summary
- Labels encoded: 8 emotion classes (0-7)
- Split: 39,244 train / 4,906 val / 4,906 test (80/10/10, stratified)
- Images resized to 224×224 and normalised (ImageNet mean/std)
- Train augmentation: random flip, rotation, colour jitter
- Class weights computed to handle severe imbalance
- DataLoaders verified: batch shape [32, 3, 224, 224]